# IBR Dynamic Modeling and Simulation

Nine teaching lessons. The solver uses only Python's standard library; ibrsim is not required. The low-frequency experiments use nominal-frequency algebraic networks with ideal current/voltage realizations, separate from the full-order source models. Keep parameters, ports, residuals and scope with every run. The website provides interactive plots.


In [ ]:
"""IBR course teaching models; Python standard library only.

Adapted conventions/control laws from PINN-IBR, reviewed 2026-10-03.
These are independently authored teaching realizations, not ibrsim models.
Network: balanced nominal-frequency algebraic impedances. GFL: ideal current
actuator + SRF PLL (4 states); droop: ideal voltage source + PQ filters (3);
VSM: adds rotor speed (4). Parallel: both branches share one PCC (7).
Switch: explicit GFL -> VSM reset, with voltage and frequency matching.
LCL: separate open-loop averaged six-state electrical plant in seconds.
No switching ripple, DC dynamics, saturation, faults, or protection.
"""
import math
import cmath

WB = 2 * math.pi * 50
NAMES = {"gfl": ["delta", "xi_pll", "id", "iq"],
         "droop": ["delta", "Pf", "Qf"],
         "vsm": ["delta", "omega", "Pf", "Qf"]}


def bounded(value, low, high, name):
    value = float(value)
    if not math.isfinite(value) or not low <= value <= high:
        raise ValueError(f"{name} must be finite and in [{low}, {high}]")
    return value


def settings(case):
    mode = case.get("mode", "droop")
    if mode not in ("frame", "lcl", "gfl", "droop", "vsm", "parallel", "switch", "compare"):
        raise ValueError("Unknown model mode")
    event = case.get("event", "p")
    if event not in ("p", "q", "v", "f"):
        raise ValueError("Unknown disturbance")
    limit = .01 if mode == "lcl" else .5 if event == "f" else .08
    return dict(mode=mode, event=event,
                step=bounded(case.get("step", .005 if mode == "lcl" else .03), -limit, limit, "step"),
                scr=bounded(case.get("scr", 5), 2, 10, "SCR"),
                mp=bounded(case.get("mp", .02), .005, .04, "m_p"),
                inertia=bounded(case.get("inertia", 4), .5, 8, "M"),
                bandwidth=bounded(case.get("bandwidth", 2), .5, 5, "PLL bandwidth"),
                angle=bounded(case.get("angle", 20), -180, 180, "frame angle"),
                dt=bounded(case.get("dt", .0005), .0001, .001, "time step"),
                duration=bounded(case.get("duration", 4), 4, 20, "duration"))


def rk4(rhs, x, h):
    a = rhs(x)
    b = rhs([v + h * d / 2 for v, d in zip(x, a)])
    c = rhs([v + h * d / 2 for v, d in zip(x, b)])
    d = rhs([v + h * e for v, e in zip(x, c)])
    return [v + h * (aa + 2 * bb + 2 * cc + dd) / 6
            for v, aa, bb, cc, dd in zip(x, a, b, c, d)]


def high_voltage_power_flow(s, z):
    """V = 1 + Z conj(S/V), choosing the high-voltage root."""
    a, b = (z * s.conjugate()).real, (z * s.conjugate()).imag
    discriminant = 1 + 4 * a - 4 * b * b
    if discriminant <= 0:
        raise ValueError("No qualified high-voltage equilibrium")
    magnitude_squared = (1 + 2 * a + math.sqrt(discriminant)) / 2
    return complex(magnitude_squared - a, b)


class TeachingCase:
    def __init__(self, kind, cfg):
        self.cfg = cfg
        self.kinds = ["gfl", "droop"] if kind == "parallel" else ["gfl" if kind == "switch" else kind]
        self.zg = complex(1, 10) / (cfg["scr"] * math.sqrt(101))
        self.zf = complex(.00625, .1)
        self.e0, self.commands, self.slices, self.x0 = [], [], [], []
        voltage = high_voltage_power_flow(complex(.6, 0), self.zg)
        current = complex(.6 / len(self.kinds), 0).conjugate() / voltage.conjugate()
        for family in self.kinds:
            start = len(self.x0)
            if family == "gfl":
                local = current * cmath.exp(-1j * cmath.phase(voltage))
                state = [cmath.phase(voltage), 0, local.real, local.imag]
                self.e0.append(1.)
                self.commands.append([.6 / len(self.kinds), 0.])
            else:
                source = voltage + self.zf * current
                state = [cmath.phase(source), .6 / len(self.kinds), 0]
                if family == "vsm":
                    state.insert(1, 1.)
                self.e0.append(1.)
                self.commands.append([.6 / len(self.kinds), (abs(source) - 1) / .0325])
            self.x0.extend(state)
            self.slices.append((start, len(self.x0)))

    def evaluate(self, x, after=False):
        cfg = self.cfg
        step = cfg["step"] if after else 0
        voltage_grid = 1 + (step if cfg["event"] == "v" else 0)
        omega_grid = 1 + (step / 50 if cfg["event"] == "f" else 0)
        dp = step / len(self.kinds) if cfg["event"] == "p" else 0
        dq = step / len(self.kinds) if cfg["event"] == "q" else 0
        forcing, denominator, entries = complex(voltage_grid), complex(1), []
        for j, (family, (start, stop)) in enumerate(zip(self.kinds, self.slices)):
            xb = x[start:stop]
            if family == "gfl":
                value = complex(xb[2], xb[3]) * cmath.exp(1j * xb[0])
                forcing += self.zg * value
            else:
                qf = xb[3] if family == "vsm" else xb[2]
                amplitude = self.e0[j] - .0325 * (qf - self.commands[j][1] - dq)
                if not .5 < amplitude < 1.5:
                    raise ValueError("Voltage source left the teaching model domain")
                value = amplitude * cmath.exp(1j * xb[0])
                forcing += self.zg * value / self.zf
                denominator += self.zg / self.zf
            entries.append(value)
        voltage = forcing / denominator
        if not .5 < abs(voltage) < 1.5:
            raise ValueError("PCC voltage left the teaching model domain")
        derivatives, currents, powers, omegas = [], [], [], []
        for j, (family, (start, stop), value) in enumerate(zip(self.kinds, self.slices, entries)):
            xb = x[start:stop]
            current = value if family == "gfl" else (value - voltage) / self.zf
            s = voltage * current.conjugate()
            p_ref, q_ref = self.commands[j][0] + dp, self.commands[j][1] + dq
            if family == "gfl":
                local = voltage * cmath.exp(-1j * xb[0])
                error = local.imag / abs(voltage)
                wn = 2 * math.pi * cfg["bandwidth"]
                omega = 1 + 2 * .707 * wn / WB * error + wn * wn / WB * xb[1]
                ref = complex(p_ref, -q_ref) / local.conjugate()
                derivative = [WB * (omega - omega_grid), error,
                              (ref.real - xb[2]) / .02, (ref.imag - xb[3]) / .02]
            elif family == "droop":
                omega = 1 - cfg["mp"] * (xb[1] - p_ref)
                derivative = [WB * (omega - omega_grid), (s.real - xb[1]) / .1, (s.imag - xb[2]) / .05]
            else:
                omega = xb[1]
                derivative = [WB * (omega - omega_grid),
                              (p_ref - xb[2] - (omega - 1) / cfg["mp"]) / cfg["inertia"],
                              (s.real - xb[2]) / .1, (s.imag - xb[3]) / .05]
            derivatives.extend(derivative)
            currents.append(current)
            powers.append(s)
            omegas.append(omega)
        current = sum(currents)
        total = voltage * current.conjugate()
        residual = abs(voltage - voltage_grid - self.zg * current)
        return dict(dx=derivatives, voltage=voltage, currents=currents, powers=powers,
                    p=total.real, q=total.imag, v=abs(voltage), frequency=50 * omegas[0],
                    residual=residual, omega=omegas[0])

    def reset_to_vsm(self, x):
        before = self.evaluate(x, True)
        source = before["voltage"] + self.zf * before["currents"][0]
        s, omega = before["powers"][0], before["omega"]
        dp = self.cfg["step"] if self.cfg["event"] == "p" else 0
        dq = self.cfg["step"] if self.cfg["event"] == "q" else 0
        self.commands[0][0] = s.real + (omega - 1) / self.cfg["mp"] - dp
        self.e0[0] = abs(source) + .0325 * (s.imag - self.commands[0][1] - dq)
        self.kinds[0] = "vsm"
        reset = [cmath.phase(source), omega, s.real, s.imag]
        after = self.evaluate(reset, True)
        error = max(abs(after["voltage"] - before["voltage"]), abs(after["p"] - before["p"]),
                    abs(after["q"] - before["q"]), abs(after["omega"] - omega))
        return reset, error


def controller_run(kind, cfg):
    model = TeachingCase(kind, cfg)
    x = model.x0[:]
    initial_residual = max(map(abs, model.evaluate(x)["dx"]))
    time, traces = [], {key: [] for key in ("p", "q", "v", "frequency")}
    if kind == "parallel":
        traces.update(p_gfl=[], p_gfm=[])
    network_residual, reset_error = 0., None
    # Events land exactly on step boundaries; each segment has frozen inputs.
    segments = [(0., 1., False), (1., 2. if kind == "switch" else cfg["duration"], True)]
    if kind == "switch":
        segments.append((2., cfg["duration"], True))
    for segment, (start, stop, after) in enumerate(segments):
        if kind == "switch" and segment == 2:
            x, reset_error = model.reset_to_vsm(x)
        count = math.ceil((stop - start) / cfg["dt"])
        h = (stop - start) / count
        stride = max(1, round(.01 / h))
        for k in range(count + 1):
            if k % stride == 0 or k == count:
                e = model.evaluate(x, after)
                network_residual = max(network_residual, e["residual"])
                time.append(stop if k == count else start + k * h)
                for key in ("p", "q", "v", "frequency"):
                    traces[key].append(e[key])
                if kind == "parallel":
                    traces["p_gfl"].append(e["powers"][0].real)
                    traces["p_gfm"].append(e["powers"][1].real)
            if k < count:
                x = rk4(lambda state: model.evaluate(state, after)["dx"], x, h)
    manifest = [f"{family}.{name}" for family in model.kinds for name in NAMES[family]]
    return dict(time=time, traces=traces, mode=kind, state_count=len(x), state_names=manifest,
                initial_residual=initial_residual, network_residual=network_residual,
                reset_error=reset_error, final_commands=model.commands,
                scope="nominal-frequency algebraic network; ideal current/voltage realization")


def frame_run(cfg):
    offset = cfg["angle"] * math.pi / 180
    time = [k / 10000 for k in range(401)]
    traces = {key: [] for key in ("va", "vb", "vc", "vd", "vq", "p", "q")}
    residual = 0.
    for t in time:
        phases = [WB * t + shift for shift in (0, -2 * math.pi / 3, 2 * math.pi / 3)]
        va = [math.sqrt(2 / 3) * math.cos(p) for p in phases]
        ia = [math.sqrt(2 / 3) * .6 * math.cos(p - math.pi / 6) for p in phases]
        vd = sum(math.sqrt(2 / 3) * math.cos(p + offset) * a for p, a in zip(phases, va))
        vq = sum(-math.sqrt(2 / 3) * math.sin(p + offset) * a for p, a in zip(phases, va))
        id_ = sum(math.sqrt(2 / 3) * math.cos(p + offset) * a for p, a in zip(phases, ia))
        iq = sum(-math.sqrt(2 / 3) * math.sin(p + offset) * a for p, a in zip(phases, ia))
        p, q = vd * id_ + vq * iq, vq * id_ - vd * iq
        residual = max(residual, abs(p - sum(v * i for v, i in zip(va, ia))))
        for key, value in zip(traces, [*va, vd, vq, p, q]):
            traces[key].append(value)
    return dict(time=time, traces=traces, mode="frame", state_count=0, initial_residual=residual,
                network_residual=0., reset_error=None, scope="balanced power-invariant Park transform")


def lcl_run(cfg):
    r1, ell1, cf = .00625, 8.4375e-5, .0008
    zg = complex(1, 10) / (cfg["scr"] * math.sqrt(101))
    r2, ell2 = .00625 + zg.real, (.1 + zg.imag) / WB
    i2 = complex(.6)
    vc = 1 + complex(r2, WB * ell2) * i2
    i1 = i2 + 1j * WB * cf * vc
    u = vc + complex(r1, WB * ell1) * i1
    x = [i1.real, i1.imag, vc.real, vc.imag, i2.real, i2.imag]
    def evaluate(state, after):
        a, v, b = complex(*state[:2]), complex(*state[2:4]), complex(*state[4:])
        command = u + (cfg["step"] if after else 0)
        da = (command - v - r1 * a) / ell1 - 1j * WB * a
        dv = (a - b) / cf - 1j * WB * v
        db = (v - 1 - r2 * b) / ell2 - 1j * WB * b
        denergy = ell1 * (a.conjugate() * da).real + cf * (v.conjugate() * dv).real + ell2 * (b.conjugate() * db).real
        balance = (command * a.conjugate()).real - b.real - r1 * abs(a)**2 - r2 * abs(b)**2
        return [da.real, da.imag, dv.real, dv.imag, db.real, db.imag], abs(denergy - balance)
    initial = max(map(abs, evaluate(x, False)[0]))
    time, traces, residual = [], {key: [] for key in ("i2d", "vcd", "vcq")}, 0.
    for start, stop, after in [(0., .01, False), (.01, .03, True)]:
        count = round((stop - start) / .000002)
        h = (stop - start) / count
        for k in range(count + 1):
            if k % 25 == 0 or k == count:
                time.append(stop if k == count else start + k * h)
                for key, value in zip(traces, [x[4], x[2], x[3]]):
                    traces[key].append(value)
                residual = max(residual, evaluate(x, after)[1])
            if k < count:
                x = rk4(lambda state: evaluate(state, after)[0], x, h)
    return dict(time=time, traces=traces, mode="lcl", state_count=6, initial_residual=initial,
                network_residual=residual, reset_error=None, scope="open-loop six-state averaged LCL; energy balance audit")


def solve(case):
    cfg = settings(case)
    if cfg["mode"] == "frame":
        result = frame_run(cfg)
    elif cfg["mode"] == "lcl":
        result = lcl_run(cfg)
    elif cfg["mode"] == "compare":
        runs = {kind: controller_run(kind, cfg) for kind in ("gfl", "droop", "vsm", "parallel")}
        first = runs["gfl"]
        result = dict(time=first["time"], traces={kind: run["traces"]["p"] for kind, run in runs.items()},
                      mode="compare", state_count=0, initial_residual=max(r["initial_residual"] for r in runs.values()),
                      network_residual=max(r["network_residual"] for r in runs.values()), reset_error=None,
                      runs={kind: {k: v for k, v in run.items() if k not in ("time", "traces")} for kind, run in runs.items()},
                      scope=first["scope"])
    else:
        result = controller_run(cfg["mode"], cfg)
    result["case"] = cfg
    return result



## C1-01 · System Boundaries and Model Representations

### Which model is sufficient for the decision?

A project team has four plausible inverter models and four different step responses. Before choosing a controller, the team must decide what each response can legitimately explain. Begin with the decision, then define the model boundary.

### Start with the consequence you need to explain

<p>A converter can be studied over switching cycles, electrical transients or slower synchronization dynamics. These are different questions about the same equipment. If the concern is switching ripple, the switching events matter. If it is an LCL oscillation, stored electrical energy matters. If it is a small power-command response over seconds, a reduced synchronization model may be enough.</p><p>The modeling decision is therefore a tradeoff between retained mechanisms and the evidence needed for a decision. State the output, disturbance and time horizon before selecting a controller family. “GFM” identifies a control concept; it does not specify model order.</p>





### Draw the terminal before counting states

<p>Trace the physical chain from average converter voltage through L₁, the capacitor, L₂ and the point of connection to the grid impedance and infinite bus. The capacitor port and PoC are separated by a branch that stores energy and dissipates power. A power value without a measurement port is incomplete.</p><p>In this course, injected current points toward the grid; positive P and Q denote injection at the declared port. The infinite bus sets voltage and frequency and the DC source is ideal. These assumptions define what the experiment can tell us about an interconnection.</p>





### Turn the boundary into a model contract

<p>Partition the model into differential states x, algebraic network variables z, inputs u and outputs y. A quantity is a state because it has an evolution law, not because it appears on a plot. The network constraint must agree with the controller at every time sample.</p><p>For the three-state droop experiment, x = [δ, P_f, Q_f]. PCC voltage and branch current are algebraic; power references and grid conditions are inputs. Capacitor voltage is absent. This immediately explains why that model cannot reproduce the six-state LCL resonance studied in Lesson 3.</p>

$$
\dot{x}=f(x,z,u),\qquad 0=g(x,z,u),\qquad y=h(x,z,u)
$$

### Work the case

A detailed droop realization retains 13 states. The browser droop realization retains 3. What did the reduction remove?

1. Count the six electrical states: two components each for i₁, v_c and i₂.
2. Count four PI integrators: two voltage-loop and two current-loop integrators.
3. The remaining angle and two power filters are the three reduced states.

13 − 3 = 10 states were removed. The reduction removes electrical storage and inner-control dynamics; it does not merely shorten a vector.

### Same terminal, different trajectories

All four families start at P = 0.6 pu and Q = 0. A +0.03 pu command at 1 s exposes their retained dynamics.



1. Before the event, all four traces agree at 0.6 pu: the terminal target has been matched.
2. After the event, their paths diverge because their retained synchronization and actuator dynamics differ.
3. The absence of LCL states means none of these four curves can explain the plant resonance.

1. Run the default comparison and identify which dynamics are retained in each trace.
2. Choose a voltage step and explain why these traces cannot establish switching-ripple accuracy.
3. Write a four-line model contract: boundary, states, inputs, measured outputs.


In [ ]:
case = {"mode": "compare", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


How many states are removed when this 13-state droop model is reduced to 3 states?

13 − 3 = 10 states were removed. The reduction removes electrical storage and inner-control dynamics; it does not merely shorten a vector.

In the three-state droop experiment, which quantity is algebraic?

1. Filtered active power
2. PCC voltage
3. Internal angle

<details><summary>Explanation</summary>

PCC voltage is obtained from the network constraint. P_f and δ have differential equations. In an explicit capacitor model, that capacitor voltage becomes a state; classification depends on the chosen representation.

</details>


Write a four-line model contract: question, port, retained states and excluded mechanisms. Use it to qualify any conclusion drawn from the four-family plot.

With the boundary fixed, the next risk is convention: two correct-looking equations can describe different quantities if coordinates and bases differ.


## C1-02 · Reference Frames and Per-Unit Conventions

### How can rotating coordinates preserve the same physics?

Two engineers report different dq currents for the same three-phase waveform. One reports positive reactive injection with positive i_q; the other uses negative i_q. Resolve the convention before blaming either controller.

### Rotate the description, preserve the power

<p>A balanced waveform becomes almost constant in a frame rotating with it. This makes electrical and control dynamics easier to interpret. The course uses a power-invariant Park transform with √(2/3) scaling, cosine d row and negative-sine q row.</p><p>On the balanced subspace, the transpose gives the inverse. Because this transform preserves the inner product, P has no additional 3/2 factor. A zero-sequence component requires another axis. Mixing transform conventions creates an error even if each borrowed equation was correct in its original setting.</p>





$$
T(\theta)=\sqrt{\frac{2}{3}}\begin{bmatrix}\cos\theta&\cos(\theta-2\pi/3)&\cos(\theta+2\pi/3)\\-\sin\theta&-\sin(\theta-2\pi/3)&-\sin(\theta+2\pi/3)\end{bmatrix}
$$

### Derive the current request from the measured voltage

<p>With injection-positive current, P = v_d i_d + v_q i_q and Q = v_q i_d − v_d i_q. If the d axis aligns with voltage, positive Q requires negative i_q. This sign is a direct consequence of the chosen q axis.</p><p>Invert both equations together to obtain the current reference. The full inverse remains valid while the PLL is misaligned, provided voltage magnitude is nonzero. Replacing it with i_d = P/v_d and i_q = −Q/v_d assumes v_q = 0 throughout the transient—a stronger assumption than the controller actually guarantees.</p>

$$
P=v_d i_d+v_q i_q,\qquad Q=v_q i_d-v_d i_q
$$

$$
\begin{bmatrix}i_d^*\\i_q^*\end{bmatrix}=\frac{1}{v_d^2+v_q^2}\begin{bmatrix}v_d&v_q\\v_q&-v_d\end{bmatrix}\begin{bmatrix}P^*\\Q^*\end{bmatrix}
$$

### Keep the dimensions underneath per unit

<p>Use 10 kVA, 400 V line-to-line RMS and 50 Hz. Then Z_b = 16 Ω, the power-invariant dq voltage base is 400 V, and the dq current base is 25 A. The line RMS current base is 14.434 A; these two current quantities must not be substituted for one another.</p><p>The dynamic coefficients ℓ = L/Z_b and c = CZ_b have units of seconds. Nominal reactance x = ω_bℓ is a different quantity. Likewise, a per-unit frequency difference needs multiplication by ω_b before it becomes an angle rate. Units are a practical way to catch plausible-looking modeling errors.</p>

$$
Z_b=\frac{V_b^2}{S_b},\quad I_{b,dq}=\frac{S_b}{V_b},\quad \dot\delta=\omega_b(\omega_{pu}-\omega_{g,pu})
$$

### Work the case

The base frequency is 50 Hz and the relative frequency difference is 0.001 pu. Determine the angle drift rate.

1. Convert the frequency difference: 50 × 0.001 = 0.05 Hz.
2. Convert cycles to radians: 2π × 0.05.
3. Use this rate in the relative-angle differential equation.

The angle rate is 0.314159 rad/s. Over one second the relative angle advances about 18°, which is large enough to matter to a synchronizing controller.

### Components change; power does not

The fixed 20° offset rotates both vectors. Read constant P and Q as the invariant, not constant voltage components across frames.



1. At +20°, voltage has a nonzero q component even though the physical voltage is unchanged.
2. Voltage and current rotate together, so their power inner product remains invariant.
3. Change the angle in the lab and check both P and Q, rather than only one coordinate component.

1. Set the frame offset to 0°, +20° and −90°; compare v_d and v_q.
2. Select the P–Q plot and verify that the physical power is invariant.
3. Edit the current lag in frame_run() and predict the signs of P and Q before running.


In [ ]:
case = {"mode": "frame", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


At 50 Hz, what angle rate corresponds to a +0.001 pu frequency difference? Enter rad/s.

The angle rate is 0.314159 rad/s. Over one second the relative angle advances about 18°, which is large enough to matter to a synchronizing controller.

At v_d = 1 and v_q = 0, which current injects Q = +0.2 pu?

1. i_q = +0.2 pu
2. i_q = 0.3 pu
3. i_q = −0.2 pu

<details><summary>Explanation</summary>

Q = −v_d i_q in this aligned frame, so i_q = −0.2. The answer follows from the stated transform and injection convention; changing either requires updating the full set of equations.

</details>


Rotate both voltage and current by 20°. Explain why v_d and v_q change while P and Q remain fixed. Record the Park normalization and current direction beside every dq plot.

Once coordinates and units are fixed, we can derive the storage equations without ambiguity. The next lesson turns the physical circuit into six electrical states.


## C1-03 · Averaged Converter and LCL Plant

### Where does an electrical oscillation come from?

A small converter-voltage step creates a fast current oscillation before slower power control can react. An ideal current source cannot explain it. Retain the LCL plant and ask how energy moves through the circuit.

### Storage determines the electrical states

<p>Each inductor current and capacitor voltage needs an evolution law. In dq coordinates, i₁, v_c and i₂ each contribute two components, giving six electrical states. The grid-side filter and grid inductance share i₂ and can be combined in series for the differential equation.</p><p>That simplification does not erase the PoC. Reconstruct its voltage across the filter branch when terminal measurements are needed. The open-loop lab steps average converter d-voltage by 0.005 pu at 10 ms; the 30 ms window and 2 μs integration step resolve the fast plant response.</p>





### Add the rotation terms with the right sign

<p>Write KVL across each inductor and KCL at the capacitor, then differentiate in the rotating frame. With J = [[0, 1], [−1, 0]], every vector gains +ω_bωJ times itself. These cross-axis terms are coordinate effects; they do not create or consume energy.</p><p>Here ℓ = L/Z_b and c = CZ_b are time coefficients. Using nominal reactance in place of ℓ would change the derivative by a factor of ω_b. Keep the plant voltage u distinct from a controller power command: control must first generate u before these equations determine current.</p>

$$
\dot i_1=\frac{u-v_c-r_1i_1}{\ell_1}+\omega_b\omega Ji_1
$$

$$
\dot v_c=\frac{i_1-i_2}{c}+\omega_b\omega Jv_c,\qquad\dot i_2=\frac{v_c-v_g-r_2i_2}{\ell_2}+\omega_b\omega Ji_2
$$

$$
v_{PoC}=v_c-r_{f2}i_2-\ell_{f2}(\dot i_2-\omega_b\omega Ji_2)
$$

### Initialize storage before evaluating the response

<p>For this plant experiment, set i₂ = 0.6 + j0 at the infinite-bus port. Compute v_c = 1 + (r₂ + jω_bℓ₂)i₂, then i₁ = i₂ + jω_bcv_c and u = v_c + (r₁ + jω_bℓ₁)i₁. These values satisfy the steady circuit together.</p><p>Setting all currents to zero would instead simulate energization. Setting i₁ = i₂ would omit capacitor current. The distinction matters: this LCL case is initialized at the infinite-bus port, whereas the later reduced-model comparison matches power at PoC.</p>

### Use energy to audit the derivation

<p>Define W = (ℓ₁|i₁|² + c|v_c|² + ℓ₂|i₂|²)/2. Differentiate it using the plant equations. Rotation terms cancel because a vector is orthogonal to its J rotation. What remains is input power minus bus export and resistive losses.</p><p>This identity checks several signs at once. A small energy-rate residual supports equation consistency; it does not validate neglected PWM or device limits. Closing the model with two power filters, one angle and four PI integrators produces the source 13-state droop realization.</p>

$$
\dot W=u^Ti_1-v_g^Ti_2-r_1\lVert i_1\rVert^2-r_2\lVert i_2\rVert^2
$$

### Work the case

At nominal speed, let v_c = [1, 0] and c = 0.0008 s. Find the steady capacitor contribution to i₁q − i₂q.

1. Set v̇_c = 0 in the capacitor equation.
2. Since J[1,0] = [0,−1], the difference i₁ − i₂ is −ω_bcJv_c.
3. The q component is +2π × 50 × 0.0008.

The contribution is +0.251327 pu. This fixed-voltage example illustrates why converter-side and grid-side current references differ.

### Watch the electrical time scale

A 0.005 pu d-voltage step at 10 ms excites the open-loop plant. Read i₂d and capacitor voltage together.



1. The 10 ms converter-voltage step is the input; the inductor current follows through an evolution law.
2. Capacitor voltage and output current respond together as energy moves between the stores.
3. Use the energy-rate identity to check signs; the plotted oscillation alone cannot audit the equations.

1. Run the +0.005 pu converter-voltage step and inspect the first 20 ms after the event.
2. Reduce SCR from 5 to 2 and rerun; explain what changed in the electrical branch.
3. Inspect lcl_run(): identify each stored-energy term and verify the energy-rate residual.


In [ ]:
case = {"mode": "lcl", "scr": 5, "step": 0.005, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


For the stated v_c and c, enter i₁q − i₂q in pu.

The contribution is +0.251327 pu. This fixed-voltage example illustrates why converter-side and grid-side current references differ.

Which states does this open-loop LCL experiment contain?

1. Two currents and one voltage, each with d and q components
2. Six controller integrators
3. Only active and reactive power

<details><summary>Explanation</summary>

The six states are i₁d, i₁q, v_cd, v_cq, i₂d and i₂q. Power is computed from electrical variables; it becomes a separate state only when an explicit measurement filter is added.

</details>


Explain an observed oscillation through stored energy and coupling. Report the energy-rate residual alongside the current curve, then identify the inner loops needed to study damping under control.

We now know what receives the converter voltage. The next question is how a GFL controller acquires its reference angle and generates the current request.


## C1-04 · PLL and Grid-Following Control

### When the grid angle moves, what does GFL follow?

The grid frequency rises by 0.1 Hz. The current controller still receives the same power request. Trace the causal chain from measured voltage, through PLL angle, to current injection before predicting the response.

### Acquire an angle before requesting current

<p>The SRF PLL expresses terminal voltage in its estimated frame and drives local v_q toward zero. Positive v_q means the voltage vector leads the estimated d axis; increasing the estimated angle closes that error. The power request is then converted into local current and rotated into the network frame.</p><p>The four-state teaching model retains relative PLL angle δ, integral ξ and two current-actuator components. A 20 ms ideal actuator replaces the current PI and LCL plant. This isolates synchronization and power tracking while removing the fast electrical dynamics from Lesson 3.</p>





### Translate a design frequency into PLL gains

<p>Normalize the detector as e = v_q/|V| and define the PI output as per-unit frequency. Multiplication by ω_b occurs in the relative-angle equation. Near alignment with a stiff voltage, the characteristic polynomial is s² + ω_bk_p s + ω_bk_i.</p><p>Matching it to s² + 2ζω_n s + ω_n² gives the gains below. The lab uses ζ = 0.707 and ω_n = 2πb. Thus b is a natural-frequency design parameter in Hz; network feedback means it need not equal the measured closed-loop −3 dB bandwidth.</p>

$$
e=\frac{v_q}{\lVert v\rVert},\quad\dot\xi=e,\quad\hat\omega=1+k_pe+k_i\xi,\quad\dot\delta=\omega_b(\hat\omega-\omega_g)
$$

$$
k_p=\frac{2\zeta\omega_n}{\omega_b},\qquad k_i=\frac{\omega_n^2}{\omega_b}
$$

### Close the feedback through the grid impedance

<p>The exact inverse P–Q map uses both measured voltage components. The network then returns V = V_g + Z_gI. Because the same voltage feeds the PLL, current injection affects the controller’s own angle measurement. Reducing SCR strengthens this interaction.</p><p>For the stated +0.1 Hz event, the PLL should track the new frequency. This teaching controller contains no frequency–watt or RoCoF power law, so frequency tracking alone does not imply a changed steady P request. Compare estimated frequency, P and PCC voltage together before drawing a conclusion.</p>

$$
\tau_i\dot i_{dq}=i_{dq}^*-i_{dq},\qquad V=V_g+Z_gI
$$

### Reconnect the physical current controller

<p>The detailed LCL realization first maps PoC power to i₂*. Capacitor-current compensation creates i₁*, and current PI with voltage feedforward and inductive decoupling generates converter voltage u. The plant equations then determine the actual current.</p><p>The full 15-state GFL has one angle, one PLL integral, two voltage measurements, one frequency measurement, two power-command filters, two current PI integrators and six electrical states. A bounded four-state response therefore cannot establish the detailed model’s inner-loop stability.</p>

$$
i_1^*=i_2^*-\omega_b\hat\omega cJv_c,\quad\dot\xi_i=i_1^*-i_1
$$

$$
u=v_c-\omega_b\hat\omega\ell_1Ji_1+K_{pi}(i_1^*-i_1)+K_{ii}\xi_i
$$

### Work the case

Use b = 2 Hz, ζ = 0.707 and f_b = 50 Hz. Calculate the proportional gain for the normalized PLL.

1. The natural angular frequency is ω_n = 2π × 2.
2. The base angular frequency is ω_b = 2π × 50.
3. Substitute into k_p = 2ζω_n/ω_b; the factors 2π cancel.

k_p = 0.05656. Changing the detector normalization or PI output units would require changing this gain.

### Frequency tracking is an angle-estimation task

For this figure, grid frequency rises 0.1 Hz at 1 s. The PLL estimate converges; the live lab below starts with a P-command baseline.



1. At 1 s, the grid frequency rises; angle error drives the PLL correction.
2. The temporary frequency overshoot belongs to the angle-estimation feedback.
3. The estimate converges to 50.1 Hz; unchanged P* supplies no frequency–watt response in this model.

1. Run the baseline P step, then select estimated frequency and PCC magnitude.
2. Apply a +0.1 Hz grid-frequency step; check the PLL frequency after the transient.
3. At SCR = 2, compare PLL design settings of 1 and 5 Hz using the same disturbance.


In [ ]:
case = {"mode": "gfl", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


For this normalized PLL, enter k_p at b = 2 Hz and ζ = 0.707.

k_p = 0.05656. Changing the detector normalization or PI output units would require changing this gain.

Local v_q is positive. Which initial PLL action is consistent with this convention?

1. Decrease the estimated angular speed
2. Increase the estimated angular speed
3. Set P* to zero automatically

<details><summary>Explanation</summary>

With positive k_p, positive v_q gives positive frequency correction, advancing the estimated d axis toward the voltage vector. This PLL equation alone does not change the power command.

</details>


At SCR = 5, predict a +0.1 Hz event before running it. Then reduce SCR to 2 and explain changes through voltage–current–PLL feedback, rather than through the GFL label alone.

GFL acquires its angle from voltage. The next controller creates its own angle from a power mismatch and closes synchronization through the network.


## C1-05 · Droop Grid-Forming Control

### How does a power mismatch create a voltage angle?

The operator raises P* by 0.03 pu. A GFM source cannot assign terminal power directly: it must change frequency, move its angle and let the network transfer more power. Follow that loop.

### The network completes the power-control loop

<p>The three-state realization retains δ, P_f and Q_f. Filtered active-power mismatch sets frequency; filtered reactive-power mismatch sets voltage magnitude. Integrating frequency creates the internal source angle. The source U = Ee^{jδ} sits behind Z_f = 0.00625 + j0.1 pu.</p><p>After a positive P* step, P_f cannot jump. Frequency initially rises, angle advances, network power increases and the filter catches up. A reference is therefore a cause of power change, not an assignment to the measured trajectory.</p>





$$
\tau_p\dot P_f=P_{PoC}-P_f,\qquad\tau_q\dot Q_f=Q_{PoC}-Q_f
$$

$$
\omega=1-m_p(P_f-P^*),\quad E=E_0-n_q(Q_f-Q^*),\quad\dot\delta=\omega_b(\omega-\omega_g)
$$

$$
I=\frac{Ee^{j\delta}-V}{Z_f},\qquad P+jQ=VI^*
$$

### Distinguish the terminal target from the raw command

<p>At a nominal-frequency locked equilibrium, P_f = P*. Reactive matching is subtler. The internal source amplitude must support the voltage drop across Z_f. With E₀ fixed at 1, Q* must compensate that required amplitude even when terminal Q is zero.</p><p>At SCR = 5, the default matched Q* is approximately 0.316795 pu while PoC Q = 0. This is a controller offset. Forcing every controller to use the same raw Q* would silently change the operating point and compromise the comparison.</p>

### A frequency event reveals the steady slope

<p>At a new locked equilibrium, internal frequency equals grid frequency. Rearranging the droop law predicts P = P* − (ω_g − 1)/m_p. A +0.1 Hz event at 50 Hz is +0.002 pu, so m_p = 0.02 predicts a −0.1 pu power change.</p><p>The same m_p also affects transient loop gain. With fixed τ_p = 0.1 s and τ_q = 0.05 s, changes in peak or damping reflect network, filter and control interactions. A steady slope alone does not determine the transient.</p>

### Restore the cascaded loops when the question requires them

<p>A detailed implementation sets v* = [E,0]. Voltage PI, output-current feedforward and capacitor decoupling create i₁*. Current PI then creates u for the LCL plant. F is the output-current feedforward gain; K_p and K_i denote the corresponding PI gains. Six electrical states, four integrators, two power filters and one angle give 13 states.</p><p>The canonical source filters power measured using v_c and i₂, whereas benchmark outputs are reconstructed at PoC. The browser lab deliberately uses PoC feedback. When reproducing the detailed source, preserve its measurement port and initialize PI integrals to support the equilibrium voltage and current; zero tracking error does not imply zero integrator state.</p><p>At a no-limit equilibrium with zero voltage and current errors, solve the two PI equations for their integrals. The nonzero offsets below support the required capacitor current and converter voltage. Setting the integrators to zero would turn an equilibrium comparison into an unintended startup transient.</p>

$$
e_v=v^*-v_c,\quad\dot\xi_v=e_v,\quad i_1^*=F i_2-\omega_b\omega cJv_c+K_{pv}e_v+K_{iv}\xi_v
$$

$$
e_i=i_1^*-i_1,\quad\dot\xi_i=e_i,\quad u=v_c-\omega_b\omega\ell_1Ji_1+K_{pi}e_i+K_{ii}\xi_i
$$

$$
\xi_v=\frac{i_1-Fi_2+\omega_b\omega cJv_c}{K_{iv}},\qquad\xi_i=\frac{u-v_c+\omega_b\omega\ell_1Ji_1}{K_{ii}}
$$

### Work the case

Let P* = 0.6 pu and m_p = 0.02. The grid rises from 50 to 50.1 Hz. Predict the locked steady power.

1. The per-unit frequency increase is 0.1/50 = 0.002.
2. The droop power reduction is 0.002/0.02 = 0.1 pu.
3. Subtract that reduction from the reference 0.6 pu.

The predicted locked power is 0.5 pu. The calculation assumes the stated model reaches the new equilibrium.

### Power changes through angle motion

The +0.03 pu P* step first creates a frequency mismatch, then increases transferred power. The transient reveals filter and network feedback.



1. At the command step, the filtered power cannot jump, so frequency changes first.
2. Advancing source angle increases power through the network; measured power feeds back.
3. The nominal-frequency equilibrium requires P = P*; inspect the window before calling it settled.

1. Run +0.03 pu P*, then inspect frequency: explain why it changes before power settles.
2. Inspect the matched Q* and compare it with measured Q at time zero.
3. Apply +0.1 Hz and compare the final P change with −0.1/(50 m_p).


In [ ]:
case = {"mode": "droop", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


For this +0.1 Hz event, enter the predicted steady P in pu.

The predicted locked power is 0.5 pu. The calculation assumes the stated model reaches the new equilibrium.

When terminal Q = 0, must this droop model also have Q* = 0?

1. Yes, for every network impedance
2. Only if the PLL gain is zero
3. No; Q* can provide the internal voltage offset required by the branch

<details><summary>Explanation</summary>

At equilibrium E = E₀ + n_q Q* when Q_f = 0. Current through Z_f requires a particular E, so Q* matches that source amplitude. A terminal target and a controller command are different quantities.

</details>


Use one command step and one frequency step to separate transient mechanism from steady slope. Explain why this droop source needs no PLL in its synchronization law.

Droop sets frequency algebraically. The next lesson adds a speed state and asks what virtual inertia changes—and what evidence it leaves unresolved.


## C1-06 · VSM and Virtual Inertia

### What does virtual inertia actually change?

A design review proposes doubling virtual inertia to improve the response. Before accepting the claim, distinguish initial rate, oscillatory behavior, steady power and physical energy capability.

### Add a speed state with an explicit evolution law

<p>The VSM replaces algebraic frequency droop with Mω̇ = P* − P_f − D(ω − 1). It retains δ, ω, P_f and Q_f. The electrical source and reactive law are the same as in the droop lab, so the comparison isolates the synchronization law.</p><p>Use M in seconds and D = 1/m_p. Under this swing convention M = 2H; the default M = 4 s corresponds to H = 2 s. The source repository separates governor and explicit damping contributions, while the teaching model uses their aggregate coefficient.</p>





$$
M\dot\omega=P^*-P_f-D(\omega-1),\qquad D=\frac1{m_p},\qquad M=2H
$$

### Separate initial rate from steady sharing

<p>Immediately after a P* step, speed and filtered power still equal their old equilibrium values. The accelerating mismatch is therefore ΔP*, so ḟ = f_bΔP*/M. Doubling M halves this initial rate under the same conditions.</p><p>At equilibrium ω̇ = 0 and ω = ω_g. The power–frequency slope then equals droop when D = 1/m_p, independent of M. Larger inertia can change oscillation and settling through its interaction with the 0.1 s power filter; it is not a general guarantee of a better trajectory.</p>

$$
\dot\delta=\omega_b(\omega-\omega_g),\qquad P_{\infty}=P^*-\frac{\omega_g-1}{m_p}
$$

$$
\dot f(1^+)=f_b\frac{\Delta P^*}{M}
$$

### Translate the control law into evidence requirements

<p>Virtual inertia describes a controller. Delivering its requested power requires a DC energy source, available current and a feasible converter voltage. The lab assumes an ideal DC supply and unlimited realization; it cannot establish those hardware capabilities.</p><p>A PLL-based GFL can also add filtered RoCoF power support while remaining grid-following. That structure differs from a VSM internal speed state. The source GFL-VI notebook includes an unstable frozen-parameter case, illustrating why an inertia-labelled term alone does not establish stability.</p>

### Work the case

A +0.03 pu power-reference step occurs at equilibrium. Use M = 4 s and a 50 Hz base to calculate initial RoCoF.

1. The speed derivative is 0.03/4 = 0.0075 pu/s.
2. Multiply by 50 Hz to convert to physical frequency rate.
3. Repeating with M = 8 s gives half the initial rate.

Initial RoCoF is 0.375 Hz/s. This local prediction says nothing by itself about settling time or energy availability.

### The same slope does not imply the same transient

Droop and VSM share source, filters and slope. Only the frequency law differs here; the 4 s window need not capture settling.



1. Both cases receive the same power command and share source, filters and steady slope.
2. VSM speed cannot jump: the added state changes the early trajectory and oscillatory response.
3. The end of a 4 s plot is a last sample, not automatically a settled value.

1. With M = 4 s, predict the initial frequency derivative for a +0.03 pu step.
2. Repeat at M = 1 and 8 s while holding m_p and SCR fixed.
3. Use the frequency-step experiment to compare the steady slope with droop GFM.


In [ ]:
case = {"mode": "vsm", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


Enter initial RoCoF for ΔP* = 0.03 pu and M = 4 s.

Initial RoCoF is 0.375 Hz/s. This local prediction says nothing by itself about settling time or energy availability.

Doubling M with fixed D and operating point does what immediately after the same P* step?

1. Halves the initial frequency derivative
2. Doubles the steady frequency–power slope
3. Removes the need for a DC energy source

<details><summary>Explanation</summary>

Initially Mω̇ = ΔP*, so ω̇ is inversely proportional to M. At equilibrium ω̇ = 0 and M disappears from the steady power–frequency relation.

</details>


Compare M = 1, 4 and 8 s with the same slope and filters. Report initial rate, peak and whether settling is visible. Extend the window to 20 s before claiming a steady value; keep growing oscillations in the report.

We have two distinct synchronization mechanisms. The next lesson couples them simultaneously at one PCC, where their interactions become part of the model.


## C1-07 · Parallel GFL–GFM Hybrid Models

### Why is a shared PCC more than two separate simulations?

A project combines a GFL branch with a droop-GFM branch. Both are active. An equal split of command increments appears simple, but a shared voltage couples their instantaneous responses.

### Agree on a common base before sharing power

<p>Both branches use the common 10 kVA system base. Each begins at P = 0.3 pu and Q = 0, producing total PoC P = 0.6 pu. A total +0.03 pu command step becomes +0.015 pu for each branch on that same base.</p><p>If physical devices have different ratings, convert their powers, currents and impedances to the common base before connecting them. Equal numerical per-unit values on different bases do not imply equal physical injections.</p>





### Solve one network seen by both controllers

<p>The GFL branch supplies I_c; the GFM branch supplies U behind Z_f. KCL gives I_g = I_c + (U − V)/Z_f. Combining it with V = V_g + Z_gI_g yields one explicit PCC voltage.</p><p>Evaluate this voltage first, then express it in each controller’s local frame and compute feedback. The seven states are the four GFL states plus three droop states, but concatenating state vectors alone does not connect the models. The common network does.</p>

$$
I_g=I_c+\frac{U-V}{Z_f},\qquad V=V_g+Z_gI_g
$$

$$
V=\frac{V_g+Z_g I_c+(Z_g/Z_f)U}{1+Z_g/Z_f}
$$

### Distinguish the sharing policy from actual sharing

<p>Compute S_c = VI_c* and S_v = VI_v* at the same PCC. Their instantaneous sum must equal total complex power. Equal command increments do not require equal transient outputs because angle and current dynamics differ.</p><p>Inspect branch and total power as SCR decreases. This is a shared-PCC current-source/voltage-source teaching model. The source REGFM_C1 notebook distinguishes an A11 positive-sequence formulation from an F21 detailed research lift; reproducing either requires its own equations and parameters.</p>

$$
S_{PCC}=V(I_c+I_v)^*=S_c+S_v
$$

### Work the case

On the common 10 kVA base, split a total +0.03 pu power-command increment equally between two branches.

1. Total physical increment is 0.03 × 10 kVA = 300 W.
2. Each branch receives 150 W.
3. On the common base, 150/10000 = 0.015 pu.

Each command increment is 0.015 pu on the common base. That does not assert equal instantaneous branch responses.

### The branches interact before their powers add

An equally split command creates distinct branch trajectories at the common PCC. Their sum follows total power at every sample. The figure shows changes from each initial power to make the small branch transients visible; the lab below displays actual powers.



1. Each branch command rises by 0.015 pu on the same system base.
2. The branch paths differ because PLL/current and droop/angle dynamics interact through V.
3. At every sample, the two branch powers sum to the total PCC power.

1. Select branch power and compare the two contributions during the +0.03 pu total step.
2. Check that the branch powers sum to the total at the event and at the final time.
3. Reduce SCR to 2, rerun, and describe which branch response changes more.


In [ ]:
case = {"mode": "parallel", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


Enter the command increment for each branch on the common base.

Each command increment is 0.015 pu on the common base. That does not assert equal instantaneous branch responses.

Why is adding independently simulated branch traces insufficient?

1. Powers cannot be added at a common terminal
2. It omits the common-PCC voltage feedback and network coupling
3. Parallel branches must have identical states

<details><summary>Explanation</summary>

Powers do add when evaluated at the same PCC. The missing part in independent simulations is the shared voltage: each branch changes it and therefore changes the other branch’s feedback.

</details>


Show that branch powers sum to total power at every sample. Then explain one unequal transient interval through the common PCC feedback and the different synchronization mechanisms.

Parallel hybrid keeps both mechanisms active. Mode-switching hybrid instead changes the active equations, making state meaning and transition policy decisive.


## C1-08 · Mode-Switching Hybrid Models

### What must remain continuous when the controller changes?

The inverter switches from GFL to VSM at 2 s, after a disturbance at 1 s. Both reduced state vectors have length four. Copying one vector into the other is tempting—and physically incorrect.

### A mode changes the meaning of the active state

<p>Between events, the active mode defines ẋ = f_m(x,u). At transition, a reset map defines x⁺ = R(x⁻,u). GFL uses [δ_PLL, ξ_PLL, i_d, i_q]; VSM uses [δ_source, ω, P_f, Q_f]. Equal dimension does not create compatible meaning.</p><p>A saved union of controller checkpoints is useful for implementation, but it is not automatically a larger ODE. Start the reset design from physical terminal quantities rather than array positions.</p>





$$
\dot x=f_m(x,u),\qquad x^+=R_{m^-\to m^+}(x^-,u)
$$

### Reconstruct the incoming source from the terminal

<p>Record V⁻, I⁻, P⁻, Q⁻ and PLL frequency just before switching. To preserve terminal current through Z_f, the incoming voltage source must be U⁺ = V⁻ + Z_fI⁻. Its source angle is arg U⁺, not necessarily the outgoing PLL angle.</p><p>Initialize VSM speed to the old estimated speed and its power filters to the measured powers. This aligns the incoming model with the outgoing terminal. The reset audit should then show terminal voltage, current, power and internal frequency jumps near roundoff.</p>

$$
U^+=V^-+Z_f I^-,\quad\delta^+=\arg U^+,\quad\omega^+=\omega^-,\quad P_f^+=P^-,\quad Q_f^+=Q^-
$$

### Make the transition policy explicit

<p>The teaching policy also chooses E₀⁺ to match source magnitude and P*⁺ = P⁻ + D(ω⁻ − 1) to make initial acceleration zero. It recalibrates the incoming command. Keeping the old command would be a different policy and could create an accelerating mismatch.</p><p>This experiment schedules one transition. A deployable design also needs guards, inactive-controller tracking, dwell rules, limits and protection. The source transition notebook treats hold/release policies and different active-state dimensions; a good endpoint response alone cannot establish a good transition.</p>

$$
E_0^+=|U^+|+n_q(Q^--Q^*),\qquad P^{*+}=P^-+D(\omega^--1)
$$

### Work the case

For a fixed reset example, take V⁻ = 1 + j0, I⁻ = 0.6 + j0 and Z_f = 0.00625 + j0.1 pu. Find the incoming source angle.

1. Multiply Z_fI⁻ = 0.00375 + j0.06.
2. Add terminal voltage: U⁺ = 1.00375 + j0.06.
3. Take atan2(0.06, 1.00375) and convert radians to degrees.

The incoming source angle is about 3.421°. The PLL can be voltage-aligned at 0° while the internal source requires a nonzero angle.

### Separate the disturbance from the reset

The disturbance is at 1 s; GFL-to-VSM switching is at 2 s. Read the terminal-compatible transition separately from the initial event.



1. The 1 s command disturbance and the 2 s mode transition are distinct events.
2. At 2 s, the incoming source reconstructs the previous terminal using U⁺ = V⁻ + Z_fI⁻.
3. Power continuity must be paired with voltage, current and frequency checks in the reset audit.

1. Run the scheduled switch and inspect the trace immediately before and after 2 s.
2. Read the reset residual and the final command offsets in the numerical audit.
3. In reset_to_vsm(), deliberately replace the reset angle by the PLL angle; inspect the resulting jump.


In [ ]:
case = {"mode": "switch", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


Enter the source angle for this fixed reset example, in degrees.

The incoming source angle is about 3.421°. The PLL can be voltage-aligned at 0° while the internal source requires a nonzero angle.

Can equal-length GFL and VSM state vectors be copied directly at a switch?

1. Yes, equal length guarantees compatibility
2. Yes, if both use per unit
3. No; reset must map physical meanings and terminal conditions

<details><summary>Explanation</summary>

The GFL integral and currents are not VSM speed and filtered powers. Reconstruct the incoming source from V and I, then map frequency and power measurements. State semantics, rather than vector length, determine the reset.

</details>


Write a continuity checklist and a command policy. Inspect the 2 s reset separately from the 1 s disturbance, then explain why angle equality is the wrong criterion here.

The final lesson puts all these interfaces into a reproducible comparison and turns curves into a defensible engineering recommendation.


## C1-09 · Equilibrium, Disturbances, and Fair Comparison

### What evidence makes a model comparison credible?

A review panel asks which controller is best. Four attractive curves are insufficient: the starting point, event, observation port and numerical accuracy must agree before the differences mean anything.

### Match the physical target, not every input number

<p>Use one 10 kVA, 400 V, 50 Hz base, one PoC and one grid. Set total initial P = 0.6 pu and Q = 0; the parallel case splits that target between branches. Solve S = VI* with V = 1 + Z_gI, selecting the high-voltage power-flow root.</p><p>Each controller must realize this point in its own coordinates. Droop and VSM need Q* offsets to support their internal voltage drop. Different raw commands can therefore produce the same physical target, which is the comparison we need.</p>





$$
S=VI^*,\qquad V=1+Z_g\overline{S/V},\qquad\lVert f(x_0,u_0)\rVert_\infty<\varepsilon
$$

$$
Z_g=\frac{1+j\,10}{\mathrm{SCR}\sqrt{101}},\qquad\Delta P^*_{branch}=\frac{\Delta P^*_{total}}{N}
$$

### Verify equilibrium and place events exactly

<p>Check max|f(x₀,u₀)| before applying a disturbance. Integrate up to 1 s under the old input, then start a new segment under the changed input. Duplicate-time samples capture before and after values without averaging a discontinuity across Runge–Kutta stages.</p><p>The default step is 0.5 ms. Repeat with 0.25 ms and compare trajectories on the same output grid. A smooth line is a rendering property; refinement is evidence about the numerical solution. Switching cases need an additional segment boundary and reset at 2 s.</p>

$$
e_h=\max_k|P_h(t_k)-P_{h/2}(t_k)|
$$

### Separate consistency, validity and validation

<p>Equilibrium, KCL, power sums and reset residuals establish consistency with the stated equations. Comparing against a higher-order realization over a declared range assesses approximation validity. Independent measurements or a qualified reference are needed for external validation.</p><p>The browser experiments provide the first kind of evidence. Retain unstable results rather than hiding them. Extend the 4 s window when VSM settling is unresolved. Conclusions should identify the model, disturbance and parameter range instead of announcing a global controller ranking.</p>

### Turn the comparison into a decision brief

<p>Submit a short brief with a model contract, matched-equilibrium table, two controlled disturbances and an explanation of the observed differences. Attach the parameter snapshot, executable experiment and a refinement check.</p><p>End with a recommendation proportional to the evidence: which realization is sufficient for the stated study, what mechanism explains its response, and which additional dynamics must be tested before extending the conclusion. The full-order source notebooks offer the next level of investigation.</p>

### Work the case

For an equilibrium illustration, required source E is 1.01 pu, E₀ = 1, terminal Q = 0 and n_q = 0.0325. Find the matching Q*.

1. At equilibrium, Q_f = terminal Q = 0.
2. Rearrange E = E₀ − n_q(Q_f − Q*).
3. Thus Q* = (1.01 − 1)/0.0325.

The matching command is 0.307692 pu despite zero terminal Q. The given source amplitude is illustrative, not the default SCR = 5 amplitude.

### Curves become evidence only after the protocol

Matched PoC power and an identical total command step isolate the stated model differences. The overlay does not establish a universal ranking.



1. All cases begin at the same total PoC P and Q, even though raw commands differ.
2. One total command step exposes the model-specific transient mechanisms.
3. Inspect equilibrium and timestep refinement before attributing a difference to control design.

1. Compare all four models at SCR = 5 with the +0.03 pu total P step.
2. Repeat with a +0.1 Hz frequency step; explain the GFL versus GFM steady responses.
3. In Python, halve case["dt"] and compare aligned samples; retain the maximum error in the report.


In [ ]:
case = {"mode": "compare", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


Enter Q* for this fixed equilibrium illustration.

The matching command is 0.307692 pu despite zero terminal Q. The given source amplitude is illustrative, not the default SCR = 5 amplitude.

What should be matched before comparing different controller families?

1. The terminal operating point, base and disturbance protocol
2. Every raw controller command
3. Only the number of states

<details><summary>Explanation</summary>

Different control laws require different internal commands to produce the same terminal P and Q. Compare declared common terminal conditions and preserve model-specific state initialization. Equal raw commands alone do not make the comparison fair.

</details>


Prepare a two-page decision brief. Give equal attention to the model contract, derivation, experiment and strength of evidence; include unresolved limits in the recommendation.

The modeling course ends here. A physics-informed or neural-ODE surrogate should inherit this same state definition, port convention, equilibrium and event protocol.
